In [30]:
# CELL 1 — INSTALL LIBRARIES

!pip install ultralytics opencv-python scikit-learn -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# CELL 2 — IMPORTS

import os
import cv2
import shutil
import random
import numpy as np

from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [32]:
# CELL 3 — DATASET PATHS

# PROJECT ROOT
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# DATASET (ONLY PINS)
source_dataset = os.path.join(project_root, "dataset", "Find pin.yolov8")

# OUTPUT DATASET
output_dataset = os.path.join(project_root, "dataset", "bowling_pins_dataset")

os.makedirs(output_dataset, exist_ok=True)

print("Paths ready")

Paths ready


In [33]:
# CELL 4 — CREATE TRAIN / VALID / TEST SPLITS

images_src = os.path.join(source_dataset, "train", "images")
labels_src = os.path.join(source_dataset, "train", "labels")

all_images = [
    f for f in os.listdir(images_src)
    if f.endswith((".jpg", ".jpeg", ".png"))
]

print("Total images:", len(all_images))

train_imgs, temp_imgs = train_test_split(
    all_images,
    test_size=0.2,
    random_state=42
)

valid_imgs, test_imgs = train_test_split(
    temp_imgs,
    test_size=0.5,
    random_state=42
)

print("Train:", len(train_imgs))
print("Valid:", len(valid_imgs))
print("Test :", len(test_imgs))

Total images: 323
Train: 258
Valid: 32
Test : 33


In [34]:
# CELL 5 — CREATE YOLO FOLDERS

splits = {
    "train": train_imgs,
    "valid": valid_imgs,
    "test": test_imgs
}

for split in splits:

    os.makedirs(
        os.path.join(output_dataset, split, "images"),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(output_dataset, split, "labels"),
        exist_ok=True
    )

print("YOLO folders created")

YOLO folders created


In [35]:
# CELL 6 — COPY IMAGES + LABELS

for split_name, image_list in splits.items():

    for img_file in image_list:

        name = os.path.splitext(img_file)[0]

        src_img = os.path.join(images_src, img_file)
        src_lbl = os.path.join(labels_src, name + ".txt")

        dst_img = os.path.join(
            output_dataset,
            split_name,
            "images",
            img_file
        )

        dst_lbl = os.path.join(
            output_dataset,
            split_name,
            "labels",
            name + ".txt"
        )

        shutil.copy(src_img, dst_img)

        if os.path.exists(src_lbl):
            shutil.copy(src_lbl, dst_lbl)

print("Dataset prepared")

Dataset prepared


In [36]:
# CELL 7 — CREATE data.yaml

yaml_text = f"""
path: {output_dataset}

train: train/images
val: valid/images
test: test/images

nc: 1

names:
  0: bowling-pin
"""

yaml_path = os.path.join(output_dataset, "data.yaml")

with open(yaml_path, "w") as f:
    f.write(yaml_text)

print("data.yaml created")
print(yaml_path)

data.yaml created
d:\Computer_Vision_Main project\dataset\bowling_pins_dataset\data.yaml


In [37]:
# CELL 8 — CHECK DATASET

for split in splits.keys():

    img_count = len(
        os.listdir(
            os.path.join(output_dataset, split, "images")
        )
    )

    lbl_count = len(
        os.listdir(
            os.path.join(output_dataset, split, "labels")
        )
    )

    print(f"\n{split.upper()}")
    print("Images:", img_count)
    print("Labels:", lbl_count)


TRAIN
Images: 258
Labels: 258

VALID
Images: 32
Labels: 32

TEST
Images: 33
Labels: 33


In [38]:
# CELL 9 — TRAIN MODEL

model = YOLO("yolov8s.pt")

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=4,
    workers=0,
    device=0,
    patience=15,
    pretrained=True,
    name="bowling_pin_only"
)

New https://pypi.org/project/ultralytics/8.4.47 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.46  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\Computer_Vision_Main project\dataset\bowling_pins_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000011BA7491F00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [39]:
# CELL 10 — VALIDATE MODEL

metrics = model.val()

print(metrics)

Ultralytics 8.4.46  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3133.21267.6 MB/s, size: 330.6 KB)
val: Scanning D:\Computer_Vision_Main project\dataset\bowling_pins_dataset\valid\labels.cache... 32 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 76, len(boxes) = 161. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
val: Fast image access  (ping: 0.00.0 ms, read: 3133.21267.6 MB/s, size: 330.6 KB)
val: Scanning D:\Computer_Vision_Main project\dataset\bowling_pins_dataset\valid\labels.cache... 32 images, 0 backgrounds, 0 corrupt

In [40]:
# CELL 11 — LOAD BEST MODEL

model = YOLO(
    os.path.join(os.getcwd(), "runs", "detect", "bowling_pin_only", "weights", "best.pt")
)

print("Best model loaded")

Best model loaded


In [43]:
# CELL 12 — VIDEO PATHS

input_video = os.path.join(project_root, "videos", "input.MOV")

output_video = os.path.join(project_root, "videos", "output_pins_only.mp4")

print(os.path.exists(input_video))

True


In [44]:
# CELL 13 — OPEN VIDEO

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise ValueError("Could not open video")

fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

out = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)

print("Video loaded")

Video loaded


In [45]:
# CELL 14 — TRACKING VARIABLES

pin_registry      = {}
pins_initialized  = False
NUM_PINS          = 0

fallen_pins       = {}
active_pins       = {}

MATCH_RADIUS_STANDING = 120
MATCH_RADIUS_FALLEN   = 400

FALL_CONFIRM      = 4
pin_baseline_wh   = {}
pin_wh_history    = {}
BASELINE_FRAMES   = 8
fall_streak       = {}

INIT_WINDOW_SECS  = 6
frame_num = 0
PIN_CLASS = 0
print("Variables ready")

Variables ready


In [46]:
# CELL 15 — DETECTION + FALL DETECTION (v11)

cap    = cv2.VideoCapture(input_video)
fps    = cap.get(cv2.CAP_PROP_FPS) or 30
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out    = cv2.VideoWriter(output_video, cv2.VideoWriter_fourcc(*'mp4v'),
                         fps, (width, height))

def cx_cy(x1,y1,x2,y2): return (x1+x2)//2, (y1+y2)//2

def get_detections(results):
    dets = []
    if (results and results[0].boxes is not None
            and results[0].boxes.id is not None):
        boxes   = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy().astype(int)
        for box, cls in zip(boxes, classes):
            if cls != PIN_CLASS: continue
            x1,y1,x2,y2 = map(int, box)
            w,h = x2-x1, y2-y1
            cx,cy = cx_cy(x1,y1,x2,y2)
            dets.append((x1,y1,x2,y2,w,h,cx,cy))
    return dets

def all_standing(dets):
    return all(d[5] > d[4] for d in dets)

def init_registry(dets):
    global NUM_PINS, pins_initialized
    for i, det in enumerate(
            sorted(dets, key=lambda d: (d[7]//120, d[6]))):
        pid = i + 1
        pin_registry[pid]   = [det[6], det[7]]
        fall_streak[pid]    = 0
        pin_wh_history[pid] = []
    NUM_PINS = len(pin_registry)
    pins_initialized = True
    print(f"Locked {NUM_PINS} pins: "
          f"{ {p: tuple(xy) for p,xy in pin_registry.items()} }")

def match_detections_to_anchors(dets):
    """
    Pass 1: match each detection to nearest anchor within radius.
    Returns (assignment dict pid->det, set of unmatched det indices).
    """
    if not dets or not pin_registry:
        return {}, set(range(len(dets)))

    pids = list(pin_registry.keys())
    costs = []
    for pid in pids:
        ax, ay = pin_registry[pid]
        radius = MATCH_RADIUS_FALLEN if pid in fallen_pins else MATCH_RADIUS_STANDING
        row = []
        for det in dets:
            d = np.sqrt((det[6]-ax)**2 + (det[7]-ay)**2)
            row.append(d if d <= radius else float('inf'))
        costs.append(row)

    assignment  = {}
    used_pids   = set()
    used_det_idx = set()

    while True:
        best = float('inf')
        bp = bd = -1
        for pi, pid in enumerate(pids):
            if pid in used_pids: continue
            for di in range(len(dets)):
                if di in used_det_idx: continue
                if costs[pi][di] < best:
                    best, bp, bd = costs[pi][di], pi, di
        if bp == -1 or best == float('inf'):
            break
        assignment[pids[bp]] = dets[bd]
        used_pids.add(pids[bp])
        used_det_idx.add(bd)

    unmatched_det_idx = set(range(len(dets))) - used_det_idx
    return assignment, unmatched_det_idx

# PASS 1: find best init frame 
print("Scanning for best initialization frame...")
init_window_frames = int(INIT_WINDOW_SECS * fps)
best_count, best_dets = 0, []

for fi in range(init_window_frames):
    ret, frame = cap.read()
    if not ret: break
    results = model.track(frame, persist=True,
                          tracker="bytetrack.yaml", conf=0.40, iou=0.3)
    dets = get_detections(results)
    if len(dets) > best_count and all_standing(dets):
        best_count, best_dets = len(dets), dets
        print(f"  frame {fi}: {best_count} standing pins")

if not best_dets:
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    for fi in range(init_window_frames):
        ret, frame = cap.read()
        if not ret: break
        results = model.track(frame, persist=True,
                              tracker="bytetrack.yaml", conf=0.40, iou=0.3)
        dets = get_detections(results)
        if len(dets) > best_count:
            best_count, best_dets = len(dets), dets

if best_dets:
    init_registry(best_dets)
print(f"Init complete — {NUM_PINS} pins registered")

# PASS 2: full video 
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    frame_num += 1
    t = frame_num / fps

    results = model.track(frame, persist=True,
                          tracker="bytetrack.yaml", conf=0.40, iou=0.3)
    annotated = frame.copy()
    dets = get_detections(results)

    # PASS 1 matching: normal radius matching
    active_pins.clear()
    assignment, unmatched_idx = match_detections_to_anchors(dets)

    for pid, det in assignment.items():
        active_pins[pid] = det
        if pid not in fallen_pins:
            cx, cy = det[6], det[7]
            pin_registry[pid][0] = int(pin_registry[pid][0]*0.88 + cx*0.12)
            pin_registry[pid][1] = int(pin_registry[pid][1]*0.88 + cy*0.12)

    # PASS 2 matching: rescue unmatched fallen-looking detections 
    # Ghost pins = registered pins with no detection this frame
    ghost_pids = [pid for pid in pin_registry if pid not in active_pins]

    # Unmatched detections that look fallen (w > h) = candidate rescue dets
    unmatched_fallen = [
        dets[i] for i in unmatched_idx
        if dets[i][4] > dets[i][5]   # w > h = looks fallen
    ]

    if ghost_pids and unmatched_fallen:
        # Match each fallen unmatched detection to nearest ghost anchor
        for det in unmatched_fallen:
            cx, cy = det[6], det[7]
            best_pid, best_dist = None, float('inf')
            for pid in ghost_pids:
                if pid in active_pins: continue   # already rescued
                ax, ay = pin_registry[pid]
                d = np.sqrt((cx-ax)**2 + (cy-ay)**2)
                if d < best_dist:
                    best_dist, best_pid = d, pid
            if best_pid is not None:
                # Rescue: assign this detection to the ghost pin
                active_pins[best_pid] = det
                # Update anchor to current position (it rolled here)
                pin_registry[best_pid][0] = int(
                    pin_registry[best_pid][0]*0.70 + cx*0.30)
                pin_registry[best_pid][1] = int(
                    pin_registry[best_pid][1]*0.70 + cy*0.30)
                # Mark fallen immediately (it's wide = lying flat)
                if best_pid not in fallen_pins:
                    fallen_pins[best_pid] = round(t, 2)

    # baselines 
    for pid, det in active_pins.items():
        if pid not in fallen_pins:
            pin_wh_history[pid].append((det[4], det[5]))

    # fall detection for normal matched pins
    for pid, (x1,y1,x2,y2,w,h,cx,cy) in active_pins.items():
        if pid in fallen_pins:
            continue

        history = pin_wh_history.get(pid, [])
        if len(history) < BASELINE_FRAMES:
            fall_streak[pid] = 0
            continue

        early  = history[:BASELINE_FRAMES]
        base_w = float(np.median([e[0] for e in early]))
        base_h = float(np.median([e[1] for e in early]))
        pin_baseline_wh[pid] = (base_w, base_h)

        ax, ay = pin_registry[pid]
        crit_wide      = w > h
        crit_shrunk_h  = base_h > 0 and h < base_h * 0.62
        crit_displaced = np.sqrt((cx-ax)**2 + (cy-ay)**2) > 55

        if crit_wide or crit_shrunk_h or crit_displaced:
            fall_streak[pid] = fall_streak.get(pid, 0) + 1
        else:
            fall_streak[pid] = max(0, fall_streak.get(pid, 0) - 1)

        if fall_streak.get(pid, 0) >= FALL_CONFIRM:
            fallen_pins[pid] = round(t - FALL_CONFIRM/fps, 2)

    # draw 
    for pid in pin_registry:
        if pid in active_pins:
            x1,y1,x2,y2,w,h,cx,cy = active_pins[pid]
            history = pin_wh_history.get(pid, [])

            if pid in fallen_pins:
                cv2.rectangle(annotated,(x1,y1),(x2,y2),(0,0,255),2)
                cv2.putText(annotated, f"Pin {pid}  FALLEN",
                            (x1,y1-22), cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,0,255),2)
                cv2.putText(annotated, f"{fallen_pins[pid]:.2f}s",
                            (x1,y2+20), cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,255),2)
            elif len(history) < BASELINE_FRAMES:
                cv2.rectangle(annotated,(x1,y1),(x2,y2),(150,150,150),1)
                cv2.putText(annotated, f"Pin {pid} ...",
                            (x1,y1-10), cv2.FONT_HERSHEY_SIMPLEX,0.45,(150,150,150),1)
            else:
                cv2.rectangle(annotated,(x1,y1),(x2,y2),(0,255,0),2)
                cv2.putText(annotated, f"Pin {pid}  {w}x{h}",
                            (x1,y1-10), cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,255,0),2)
        else:
            # ghost — no detection found anywhere for this pin
            ax = int(pin_registry[pid][0])
            ay = int(pin_registry[pid][1])
            color = (0,0,180) if pid in fallen_pins else (80,80,80)
            cv2.circle(annotated,(ax,ay),20,color,2)
            cv2.putText(annotated, f"Pin {pid}",
                        (ax-20,ay-25), cv2.FONT_HERSHEY_SIMPLEX,0.45,color,1)

    cv2.putText(annotated, f"Pins Fallen: {len(fallen_pins)}",
                (20,40), cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,255),2)
    cv2.putText(annotated, f"Time: {t:.2f}s",
                (20,80), cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255),2)
    out.write(annotated)

cap.release()
out.release()
print(f"Done — {len(fallen_pins)} / {NUM_PINS} pins fallen")

Scanning for best initialization frame...


0: 384x640 5 bowling-pins, 78.7ms
Speed: 11.1ms preprocess, 78.7ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
  frame 0: 5 standing pins

0: 384x640 5 bowling-pins, 78.7ms
Speed: 11.1ms preprocess, 78.7ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
  frame 0: 5 standing pins

0: 384x640 5 bowling-pins, 49.3ms
Speed: 3.7ms preprocess, 49.3ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 bowling-pins, 49.3ms
Speed: 3.7ms preprocess, 49.3ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 bowling-pins, 35.4ms
Speed: 2.6ms preprocess, 35.4ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 bowling-pins, 35.4ms
Speed: 2.6ms preprocess, 35.4ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 bowling-pins, 35.4ms
Speed: 3.0ms preprocess, 35.4ms inference, 2.2ms postprocess per image at 

In [47]:
# CELL 16 — FINAL SUMMARY SCREEN

summary = np.zeros((height, width, 3), dtype=np.uint8)

cv2.putText(
    summary,
    "FINAL RESULT",
    (width // 3, height // 3),
    cv2.FONT_HERSHEY_SIMPLEX,
    1.5,
    (0,255,255),
    3
)

cv2.putText(
    summary,
    f"Pins Fallen: {len(fallen_pins)}",
    (width // 4, height // 2),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (255,255,255),
    2
)

cv2.putText(
    summary,
    f"Total Time: {frame_num/fps:.2f}s",
    (width // 4, height // 2 + 50),
    cv2.FONT_HERSHEY_SIMPLEX,
    1,
    (255,255,255),
    2
)

for _ in range(int(fps * 2)):
    out.write(summary)

cap.release()
out.release()

cv2.destroyAllWindows()

print("DONE")
print("Saved to:", output_video)

DONE
Saved to: d:\Computer_Vision_Main project\videos\output_pins_only.mp4
